In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from scipy.interpolate import interp1d

In [2]:
df = pd.read_csv("paired_simulation_labels_combined.csv")

In [3]:
# ==========================================
# 1. PyTorch Dataset & Interpolation
# ==========================================

class LightCurveDataset(Dataset):
    def __init__(self, df, max_length=6000):
        """
        Loads light curves directly from disk, centers them, 
        and standardizes their length using interpolation.
        """
        self.df = df.reset_index(drop=True)
        self.max_length = max_length
        self.event_map = {"neither": 0, "flare": 1, "transit": 2, "both": 3}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sys_id = row['system_id']
        label_str = row.get('event_class', 'neither') # Ensure you mapped this in your df
        label_idx = self.event_map[label_str]
        
        # Load the corrected light curve
        folder_path = f"outputs/{sys_id}/"
        dat_files = glob.glob(os.path.join(folder_path, "*_driftcorrected.dat"))
        if not dat_files:
            dat_files = glob.glob(os.path.join(folder_path, "*.dat"))
            
        try:
            data = np.genfromtxt(dat_files[0])
            time = data[:, 0]
            flux = data[:, 1]
            
            # Median center the flux
            flux = flux - np.nanmedian(flux)
            
            # Standardize length to exactly `max_length` via interpolation
            f_interp = interp1d(np.linspace(0, 1, len(flux)), flux, kind='linear')
            uniform_flux = f_interp(np.linspace(0, 1, self.max_length))
            
        except Exception:
            # Fallback for corrupted/missing files: return zeros
            uniform_flux = np.zeros(self.max_length)

        # PyTorch Conv1D expects shape: [Channels, Length] -> [1, max_length]
        flux_tensor = torch.tensor(uniform_flux, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(label_idx, dtype=torch.long)
        
        return flux_tensor, label_tensor

In [4]:
# ==========================================
# 2. The 1D-CNN Architecture
# ==========================================

class AstroNet1D(nn.Module):
    def __init__(self, num_classes=4):
        super(AstroNet1D, self).__init__()
        
        # Block 1: Feature Extraction (capturing broad trends)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=15, stride=2, padding=7)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # Block 2: Capturing finer details (like transit dips)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=11, stride=1, padding=5)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # Block 3: Deep features
        self.conv3 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, stride=1, padding=3)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # Calculate the flattened size dynamically 
        # (6000 -> conv1(3000) -> pool1(750) -> pool2(187) -> pool3(46) * 64 channels = 2944)
        self.flatten = nn.Flatten()
        
        # Fully Connected Classifier
        self.fc1 = nn.Linear(64 * 46, 128)
        self.dropout = nn.Dropout(0.5)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flatten(x)
        x = self.relu4(self.dropout(self.fc1(x)))
        x = self.fc2(x) # No softmax here; CrossEntropyLoss applies it automatically
        return x



In [ ]:
# ==========================================
# 3. Training & Evaluation Pipeline
# ==========================================

# 1. Setup Device (Will use Apple Silicon MPS if available, otherwise CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Prepare Data Loaders (Assuming 'df' has the 'event_class' mapped)
# Map your anomaly classes to strings if you haven't already:
event_labels = {0: "neither", 1: "flare", 2: "transit", 3: "both"}
df['event_class'] = df['anomaly_class'].map(event_labels)

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['event_class'])

train_dataset = LightCurveDataset(train_df, max_length=6000)
test_dataset = LightCurveDataset(test_df, max_length=6000)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Initialize Model, Loss, and Optimizer
model = AstroNet1D(num_classes=4).to(device)

# Using class weights is crucial because transits/both are usually rarer
# You can calculate exact inverse frequencies, or approximate:
class_weights = torch.tensor([1.0, 1.0, 1.0, 1.0]).to(device) 
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training Loop
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_flux, batch_labels in train_loader:
        batch_flux, batch_labels = batch_flux.to(device), batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_flux)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Training Loss: {running_loss/len(train_loader):.4f}")

Training on device: mps


NameError: name 'df' is not defined

In [ ]:
# 5. Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_flux, batch_labels in test_loader:
        batch_flux, batch_labels = batch_flux.to(device), batch_labels.to(device)
        outputs = model(batch_flux)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

# Convert numerical predictions back to string labels for the confusion matrix
reverse_map = {v: k for k, v in train_dataset.event_map.items()}
pred_strings = [reverse_map[p] for p in all_preds]
label_strings = [reverse_map[l] for l in all_labels]
class_names = ["neither", "flare", "transit", "both"]

print("\n1D-CNN Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(label_strings, pred_strings, labels=class_names),
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names]
))

print("\n1D-CNN Classification Report:")
print(classification_report(label_strings, pred_strings, labels=class_names))